In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
THR_INCIDENT_WEEK=4

In [12]:
df = pd.read_csv("Analysis/DTW_Incidents/dtw_incident_daily_predicted_incident_window3.csv")
df.head()

In [13]:
df = df.sort_values(by=["account_name", "audience_id", "date"], ascending=True).reset_index(drop=True)
df

In [15]:
df["month"] = pd.to_datetime(df["date"]).dt.month
df["week"] = pd.to_datetime(df["date"]).dt.isocalendar().week
df.head()

In [16]:
incidents_by_week = df.groupby(by=["account_name", "audience_id", "week"])["is_incident"].sum().reset_index().rename(columns={"is_incident": "incident_by_week"})
incidents_by_week

In [17]:
df = pd.merge(df, incidents_by_week[["account_name", "audience_id", "week", "incident_by_week"]], how="left", on=["account_name", "audience_id", "week"])
df["incident_by_week_thr"] = df["incident_by_week"] >= THR_INCIDENT_WEEK
df

# Check Weekly Incidents

In [18]:
df_week = df.drop_duplicates(subset=["account_name", "audience_id", "week"])[["account_name", "audience_id", "model_type","week", "month", "incident_by_week_thr"]]
df_week

## Total Incidents by Week

In [19]:
total_incidents = df_week.groupby(by=["week","model_type"])["incident_by_week_thr"].sum().rename("total_incidents")
total_tracked_measurements = df_week.groupby(by=["week","model_type"])["incident_by_week_thr"].count().rename("total_tracked_measurements")
total_incidents = pd.concat([total_incidents, total_tracked_measurements], axis=1).reset_index()
total_incidents["incidents_in_%"] = total_incidents["total_incidents"] / total_incidents["total_tracked_measurements"] * 100
total_incidents

In [20]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 1, 1)
ax.set_title("Incidents in % Over Time")
sns.barplot(
    data=total_incidents,
    x="week",
    y="incidents_in_%",
    hue="model_type",
    # marker="o",
    ax=ax,
    errorbar=None
)
ax.set_xlabel("week")
ax.set_ylabel("incidents in %")
plt.xticks(rotation=0)
plt.grid(True)
plt.tight_layout()
plt.show()

# Incidents By Customer

In [21]:
df_week_workspace = df_week.groupby(by=["account_name","week","model_type","month"])["incident_by_week_thr"].sum().rename("total_incidents")
total_tracked_measurements_workspace = df_week.groupby(by=["account_name", "week","model_type", "month"])["incident_by_week_thr"].count().rename("total_tracked_measurements")
df_week_workspace = pd.concat([df_week_workspace, total_tracked_measurements_workspace], axis=1).reset_index()
df_week_workspace["incidents_in_%"] = df_week_workspace["total_incidents"] / df_week_workspace["total_tracked_measurements"] * 100
df_week_workspace

In [22]:
df_month = df_week_workspace.groupby(by=["account_name","month","model_type"])["total_incidents"].sum().rename("total_incidents")
total_tracked_measurements_month = df_week_workspace.groupby(by=["account_name", "month","model_type"])["total_tracked_measurements"].sum().rename("total_tracked_measurements")
df_month = pd.concat([df_month, total_tracked_measurements_month], axis=1).reset_index()
df_month["incidents_in_%"] = df_month["total_incidents"] / df_month["total_tracked_measurements"] * 100
df_month

In [23]:
stats_month = df_month.groupby(by=["month","model_type"])["incidents_in_%"].describe().reset_index().sort_values(by=["model_type","month"])
stats_month

# Total incidents with excluded customers

In [24]:
exclude_customer = ["Asambeauty"]
total_incidents_exclude_workspaces = df_week[df_week["account_name"].isin(exclude_customer) == False][["week", "model_type", "incident_by_week_thr"]].groupby(by=["week","model_type"])["incident_by_week_thr"].sum().rename("total_incidents")
total_tracked_measurements_exclude_workspaces = df_week[df_week["account_name"].isin(exclude_customer) == False].groupby(by=["week","model_type"])["incident_by_week_thr"].count().rename("total_tracked_measurements")
total_incidents_exclude_workspaces = pd.concat([total_incidents_exclude_workspaces, total_tracked_measurements_exclude_workspaces], axis=1).reset_index()
total_incidents_exclude_workspaces["incidents_in_%"] = total_incidents_exclude_workspaces["total_incidents"] / total_incidents_exclude_workspaces["total_tracked_measurements"] * 100
total_incidents_exclude_workspaces

In [25]:
stats_month_excluded = df_month[df_month["account_name"].isin(exclude_customer) == False].groupby(by=["month","model_type"])["incidents_in_%"].describe().reset_index().sort_values(by=["model_type","month"])
stats_month_excluded

In [26]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 1, 1)
ax.set_title(f"Incidents in % Over Time without {exclude_customer}")
sns.barplot(
    data=total_incidents,
    x="week",
    y="incidents_in_%",
    hue="model_type",
    # marker="o",
    ax=ax,
    errorbar=None
)
ax.set_xlabel("week")
ax.set_ylabel("incidents in %")
plt.xticks(rotation=0)
plt.grid(True)
plt.tight_layout()
plt.show()

In [27]:
df.groupby("month")["audience_id"].nunique()